# Prueba de extremo a extremo

Lo que la web manda al servidor, y lo que el servidor devuelve.

1. **ENTRADAS** — los datos del formulario
2. **LLAMADA** — se pasan al script del servidor, que escribe en la base
3. **SALIDAS** — lo que se lee de vuelta

Nada mas.

In [1]:
import sys, os, subprocess, json
from pathlib import Path
from datetime import date, timedelta
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "production" / "app"))
os.environ.setdefault("TFM_EMAIL", "acjg.sgs@outlook.com")
pd.set_option("display.width", 200, "display.max_columns", 60)

from caso import conexion
con = conexion()
print(f"  usuario  {os.environ['TFM_EMAIL']}")
print(f"  base     conectada")

  usuario  acjg.sgs@outlook.com
  base     conectada


---
## 1 · ENTRADAS

**Un año completo de fichero basta**, tanto en consumo como en generación. El fichero solo
aporta la **forma**: se resume en 576 valores —12 meses x laborable/finde x 24 horas— y el
tamaño lo pone aparte el campo `consumo_anual_mwh` (o `potencia_pico_kwp`). Por eso el mismo
fichero sirve para una instalación de 200 MWh/año y para otra de 2.000.

Si se suben **varios años**, se promedian en esa misma rejilla: la forma sale más estable y
un año raro pesa menos. No es necesario, pero no estorba.

Lo que sí importa es que **el año esté completo**. Con medio año, los meses que faltan se
rellenan con la media de los demás, y en una fábrica con parón de agosto o en una fotovoltaica
eso deforma el resultado.

In [2]:
# ── LO QUE SUBE EL USUARIO. Es todo lo que la web pregunta ────────────────────
ENTRADAS = {
    "bateria": {
        "code": "PRUEBA-BAT", "name": "LFP 100 kW / 4 h",
        "potencia_kw": 100, "duracion_h": 4,
        "eficiencia": 0.90, "soc_min": 0.05, "soc_max": 0.95,
        "ciclos_vida": 6000, "capex_eur_mwh": 200_000,
    },
    "consumo": {
        "code": "PRUEBA-CON", "name": "Fabrica",
        "fichero": "docs/plantillas/plantilla_consumo.csv", "unidad": "kwh",
        "consumo_anual_mwh": 350,
        "recargo_eur_mwh": 70, "precio_excedente_pct": 80,
    },
    "generacion": {
        "code": "PRUEBA-GEN", "name": "FV 250 kWp",
        "fichero": "docs/plantillas/plantilla_generacion.csv", "unidad": "kwh",
        "potencia_pico_kwp": 250,
    },
}

for bloque, campos in ENTRADAS.items():
    print(f"\n  {bloque.upper()}")
    for k, v in campos.items():
        print(f"    {k:24s} {v}")


  BATERIA
    code                     PRUEBA-BAT
    name                     LFP 100 kW / 4 h
    potencia_kw              100
    duracion_h               4
    eficiencia               0.9
    soc_min                  0.05
    soc_max                  0.95
    ciclos_vida              6000
    capex_eur_mwh            200000

  CONSUMO
    code                     PRUEBA-CON
    name                     Fabrica
    fichero                  docs/plantillas/plantilla_consumo.csv
    unidad                   kwh
    consumo_anual_mwh        350
    recargo_eur_mwh          70
    precio_excedente_pct     80

  GENERACION
    code                     PRUEBA-GEN
    name                     FV 250 kWp
    fichero                  docs/plantillas/plantilla_generacion.csv
    unidad                   kwh
    potencia_pico_kwp        250


### Lo que el servidor decide solo

El estudio **no se configura**. El usuario sube sus datos y obtiene un resultado; no elige
horizonte, ni política de carga, ni número de escenarios. Todo eso lo fija el servidor y se
deja escrito aquí para que se sepa **con qué supuestos** salió el número — no para que nadie
los toque desde la web.

Las fechas salen de la propia curva publicada: se estudia todo lo que la curva cubre, que es
el horizonte para el que hay precio.

In [3]:
# ── LO QUE FIJA EL SERVIDOR. No aparece en el formulario ──────────────────────
c0 = pd.read_sql("SELECT date_from, date_to, n_scenarios, generated_at FROM app_curve",
                 con).iloc[0]

FIJO = {
    "modo":            "autoconsumo",   # hay consumo y generacion: no hay otra opcion
    "politica_carga":  "libre",
    "desde":           str(c0.date_from),
    "hasta":           str(c0.date_to),
    "ventana_dias":    7,
    "tasa_descuento":  0.07,
    "opex_pct":        0.015,
    "crecimiento_pct": 1.0,             # del consumo, anual
    "degradacion_pct": 0.5,             # de la generacion, anual
    "escenarios":      20,
    "coste_ciclo":     None,            # sale de la ficha de la bateria
}

for k, v in FIJO.items():
    print(f"    {k:20s} {'(de la ficha)' if v is None else v}")
print(f"\n  el horizonte son los {(pd.Timestamp(c0.date_to) - pd.Timestamp(c0.date_from)).days:,}"
      f" dias que cubre la curva del {c0.generated_at:%Y-%m-%d}".replace(",", "."))

    modo                 autoconsumo
    politica_carga       libre
    desde                2026-09-02
    hasta                2046-12-31
    ventana_dias         7
    tasa_descuento       0.07
    opex_pct             0.015
    crecimiento_pct      1.0
    degradacion_pct      0.5
    escenarios           20
    coste_ciclo          (de la ficha)

  el horizonte son los 7.425 dias que cubre la curva del 2026-08-31


C:\Users\torgi\AppData\Local\Temp\ipykernel_50368\3439330316.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  c0 = pd.read_sql("SELECT date_from, date_to, n_scenarios, generated_at FROM app_curve",


---
## 2 · LLAMADA AL SERVIDOR

Cinco ordenes a `production/app/caso.py`. Las tres primeras dan de alta la bateria y las dos
instalaciones, la cuarta crea el caso y la quinta lo ejecuta.

In [4]:
b, c_, g = (ENTRADAS[k] for k in ("bateria", "consumo", "generacion"))
CODIGO = "PRUEBA-01"

ORDENES = [
    ["bateria", "--code", b["code"], "--nombre", b["name"],
     "--potencia", str(b["potencia_kw"] / 1000), "--duracion", str(b["duracion_h"]),
     "--eficiencia", str(b["eficiencia"]), "--soc-min", str(b["soc_min"]),
     "--soc-max", str(b["soc_max"]), "--ciclos", str(b["ciclos_vida"]),
     "--capex", str(b["capex_eur_mwh"])],

    ["consumo", "--code", c_["code"], "--nombre", c_["name"],
     "--fichero", c_["fichero"], "--unidad", c_["unidad"],
     "--anual", str(c_["consumo_anual_mwh"]),
     "--recargo", str(c_["recargo_eur_mwh"]),
     "--excedente-pct", str(c_["precio_excedente_pct"]),
     "--crecimiento", str(FIJO["crecimiento_pct"])],

    ["generacion", "--code", g["code"], "--nombre", g["name"],
     "--fichero", g["fichero"], "--unidad", g["unidad"],
     "--mwp", str(g["potencia_pico_kwp"] / 1000),
     "--degradacion", str(FIJO["degradacion_pct"])],

    ["crear", "--code", CODIGO, "--nombre", "Estudio", "--modo", FIJO["modo"],
     "--bateria", b["code"], "--consumo", c_["code"], "--generacion", g["code"],
     "--desde", FIJO["desde"], "--hasta", FIJO["hasta"],
     "--politica", FIJO["politica_carga"], "--ventana", str(FIJO["ventana_dias"]),
     "--tasa", str(FIJO["tasa_descuento"]), "--opex", str(FIJO["opex_pct"])],

    ["ejecutar", "--code", CODIGO, "--escenarios", str(FIJO["escenarios"])],
]

for orden in ORDENES:
    print(f"\n$ python production/app/caso.py {' '.join(orden)}")
    r = subprocess.run([sys.executable, "production/app/caso.py", *orden],
                       cwd=REPO, capture_output=True, text=True,
                       encoding="utf-8", errors="replace")
    for l in (r.stdout or "").splitlines():
        if l.strip() and "escenario " not in l:
            print(f"  {l}")
    if r.returncode:
        print((r.stderr or "")[-1500:])
        raise SystemExit("ha fallado")


$ python production/app/caso.py bateria --code PRUEBA-BAT --nombre LFP 100 kW / 4 h --potencia 0.1 --duracion 4 --eficiencia 0.9 --soc-min 0.05 --soc-max 0.95 --ciclos 6000 --capex 200000


    bateria 'PRUEBA-BAT' (id 5) · 0.4 MWh · 0.36 utiles
    coste de ciclo: 37.0 EUR/MWh descargado

$ python production/app/caso.py consumo --code PRUEBA-CON --nombre Fabrica --fichero docs/plantillas/plantilla_consumo.csv --unidad kwh --anual 350 --recargo 70 --excedente-pct 80 --crecimiento 1.0


    formato LARGO: fecha='fecha' hora='hora' valor='consumo_kwh'
    horas numeradas 1..25 (convencion española): se pasa a 0..23
       28-03-2027: el calendario tiene 23 horas · cambio de hora de marzo, no existe la 2:00
       31-10-2027: el calendario tiene 25 horas · cambio de hora de octubre, 2A y 2B a la misma hora
    8,760 horas · 365 dias · 2027-01-01 -> 2027-12-31
    1 filas duplicadas promediadas (hora repetida de octubre)
    1 horas ausentes interpoladas dentro de su dia
    consumo 'PRUEBA-CON' (id 2) · 350 MWh/año · 576 filas de forma
    tarifa: +70 EUR/MWh al importar · 80% del spot al exportar

$ python production/app/caso.py generacion --code PRUEBA-GEN --nombre FV 250 kWp --fichero docs/plantillas/plantilla_generacion.csv --unidad kwh --mwp 0.25 --degradacion 0.5


    formato LARGO: fecha='fecha' hora='hora' valor='generacion_kwh'
    horas numeradas 1..25 (convencion española): se pasa a 0..23
       28-03-2027: el calendario tiene 23 horas · cambio de hora de marzo, no existe la 2:00
       31-10-2027: el calendario tiene 25 horas · cambio de hora de octubre, 2A y 2B a la misma hora
    8,760 horas · 365 dias · 2027-01-01 -> 2027-12-31
    1 filas duplicadas promediadas (hora repetida de octubre)
    1 horas ausentes interpoladas dentro de su dia
    generacion 'PRUEBA-GEN' (id 5) · 0.250 MWp · 576 filas de forma

$ python production/app/caso.py crear --code PRUEBA-01 --nombre Estudio --modo autoconsumo --bateria PRUEBA-BAT --consumo PRUEBA-CON --generacion PRUEBA-GEN --desde 2026-09-02 --hasta 2046-12-31 --politica libre --ventana 7 --tasa 0.07 --opex 0.015


    caso 'PRUEBA-01' (id 8) · autoconsumo · 2026-09-02 -> 2046-12-31

$ python production/app/caso.py ejecutar --code PRUEBA-01 --escenarios 20


    caso 'PRUEBA-01' · autoconsumo · 2026-09-02 -> 2046-12-31
    bateria 'PRUEBA-BAT': 0.1 MW / 4 h · coste de ciclo 37.0 EUR/MWh
    precio: simulado 7426 dias
    frontera en 2026-09-02: antes hay una realizacion, despues 20
    consumo 'PRUEBA-CON': 350 MWh/año · +70 EUR/MWh al importar
    generacion 'PRUEBA-GEN': 0.250 MWp
    resuelto en 249s
    despacho horario de 3 escenarios: 534,672 filas
    guardado en app_case_run id 13 · 21 años
    vida util 8 años · 0.96 ciclos/dia
    VAN al 7%: P10 -1,474 · P50 14 · P90 1,455 EUR
    escenarios con VAN positivo: 50% · cobertura del CAPEX 188%
          origen  dias    media      p10      p50      p90  ciclos
  año                                                             
  2026  simulado   121   4328.4   3936.3   4301.5   4656.5     0.9
  2027  simulado   365  15096.8  14494.7  15163.2  15704.8     1.0
  2028  simulado   366  14824.1  14132.3  14868.6  15592.3     1.0
  2029  simulado   365  14765.3  14022.1  14787.2  15548.5    

---
## 3 · SALIDAS

Lo que queda en la base y la web tiene que enseñar.

In [5]:
r = pd.read_sql("""
    SELECT r.* FROM app_case_run r
    JOIN app_study_case c ON c.case_id = r.case_id
    JOIN app_user u ON u.user_id = c.user_id
    WHERE u.email = %(e)s AND c.code = %(c)s
    ORDER BY r.run_at DESC LIMIT 1""",
    con, params={"e": os.environ["TFM_EMAIL"], "c": CODIGO}).iloc[0]

ETIQUETAS = [
    ("margin_annual_mean",  "margen anual por MW instalado", "EUR/año/MW"),
    ("savings_vs_no_batt",  "ahorro frente a no tener bateria", "EUR"),
    ("cycles_per_day",      "ciclos al dia", ""),
    ("life_years",          "vida por ciclado", "años"),
    ("npv_p10",             "VAN · decil malo", "EUR"),
    ("npv_p50",             "VAN · mediana", "EUR"),
    ("npv_p90",             "VAN · decil bueno", "EUR"),
    ("npv_positive_pct",    "escenarios con VAN positivo", "%"),
    ("capex_coverage_pct",  "cobertura del CAPEX", "%"),
    ("days_historical",     "dias con precio real", ""),
    ("days_simulated",      "dias simulados", ""),
    ("n_scenarios",         "escenarios", ""),
    ("solver",              "solver", ""),
    ("solver_seconds",      "tiempo de calculo", "s"),
    ("curve_generated_at",  "curva usada", ""),
]
def es(v, dec=2):
    """Formato español. Un `.replace(',', '.')` a secas deja 144.844.23, que no es un numero:
    hay que intercambiar los dos separadores, no sustituir uno."""
    return f"{v:,.{dec}f}".replace(",", "\x00").replace(".", ",").replace("\x00", ".")

print("  RESULTADO\n")
for col, txt, uni in ETIQUETAS:
    v = r[col]
    s = "-" if pd.isna(v) else es(v) if isinstance(v, float) else str(v)[:19]
    print(f"    {txt:34s} {s:>18s} {uni}")

  RESULTADO

    margen anual por MW instalado              141.257,58 EUR/año/MW
    ahorro frente a no tener bateria           287.391,47 EUR
    ciclos al dia                                    0,96 
    vida por ciclado                                 8,00 años
    VAN · decil malo                            -1.474,46 EUR
    VAN · mediana                                   14,34 EUR
    VAN · decil bueno                            1.455,24 EUR
    escenarios con VAN positivo                     50,00 %
    cobertura del CAPEX                            187,92 %
    dias con precio real                                0 
    dias simulados                                   7426 
    escenarios                                         20 
    solver                                       highs-lp 
    tiempo de calculo                              249,28 s
    curva usada                        2026-08-31 20:55:52 


C:\Users\torgi\AppData\Local\Temp\ipykernel_50368\2749889169.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  r = pd.read_sql("""


In [6]:
a = pd.read_sql("""
    SELECT year, origin, days, margin_mean, p10, p50, p90, cycles_per_day,
           energy_charged_mwh, energy_discharged_mwh, grid_import_mwh, grid_export_mwh
    FROM app_case_result_annual WHERE run_id = %(r)s ORDER BY year""",
    con, params={"r": int(r.run_id)})
print(f"  POR AÑO · {len(a)} filas\n")
display(a.round(1))

  POR AÑO · 21 filas



C:\Users\torgi\AppData\Local\Temp\ipykernel_50368\1958771393.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  a = pd.read_sql("""


,year,origin,days,margin_mean,p10,p50,p90,cycles_per_day,energy_charged_mwh,energy_discharged_mwh,grid_import_mwh,grid_export_mwh
0,2026,simulado,121,4328.4,3936.3,4301.5,4656.5,0.9,45.8,41.2,53.7,16.7
1,2027,simulado,365,15096.8,14494.7,15163.2,15704.8,1.0,141.8,127.6,128.1,158.4
2,2028,simulado,366,14824.1,14132.3,14868.6,15592.3,1.0,141.5,127.2,130.5,155.9
3,2029,simulado,365,14765.3,14022.1,14787.2,15548.5,1.0,140.5,126.5,133.3,152.8
4,2030,simulado,365,14694.2,14037.1,14742.5,15120.8,1.0,140.0,126.0,136.8,150.7
5,2031,simulado,365,14292.7,13397.7,14244.5,15113.6,1.0,139.5,125.5,142.6,150.7
6,2032,simulado,366,14206.5,13415.3,14298.0,14904.8,1.0,140.3,126.3,146.5,148.4
7,2033,simulado,365,14328.6,13697.9,14422.2,14884.4,1.0,141.4,127.2,145.1,142.6
8,2034,simulado,365,14153.0,13203.9,14174.1,15000.7,1.0,140.3,126.2,147.8,139.7
9,2035,simulado,365,13947.7,13249.8,14063.1,14513.5,1.0,139.6,125.7,151.3,137.0


In [7]:
# el escenario 0 solo existe en el tramo historico; si el caso es todo futuro, los
# escenarios guardados son otros, asi que se coge el primero que haya en vez de fijar el 0
d = pd.read_sql("""
    SELECT datetime, price, charge_mw, discharge_mw, soc_mwh,
           grid_import_mwh, grid_export_mwh, load_mwh, generation_mwh
    FROM app_case_dispatch WHERE run_id = %(r)s
      AND scenario = (SELECT min(scenario) FROM app_case_dispatch WHERE run_id = %(r)s)
    ORDER BY datetime LIMIT 24""", con, params={"r": int(r.run_id)})
if d.empty:
    print("  el caso se ejecuto con --sin-despacho: no hay detalle horario")
else:
    n = pd.read_sql("SELECT count(*) n FROM app_case_dispatch WHERE run_id = %(r)s",
                    con, params={"r": int(r.run_id)}).n[0]
    print(f"  HORA A HORA · {n:,} filas guardadas · primeras 24\n".replace(",", "."))
    display(d.round(3))

C:\Users\torgi\AppData\Local\Temp\ipykernel_50368\2515481035.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  d = pd.read_sql("""


C:\Users\torgi\AppData\Local\Temp\ipykernel_50368\2515481035.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  n = pd.read_sql("SELECT count(*) n FROM app_case_dispatch WHERE run_id = %(r)s",


  HORA A HORA · 534.672 filas guardadas · primeras 24



,datetime,price,charge_mw,discharge_mw,soc_mwh,grid_import_mwh,grid_export_mwh,load_mwh,generation_mwh
0,2026-09-02 00:00:00,157.092,0.000,0.000,0.020,0.033,0.000,0.033,0.000
1,2026-09-02 01:00:00,137.587,0.000,0.000,0.020,0.033,0.000,0.033,0.000
2,2026-09-02 02:00:00,148.119,0.000,0.000,0.020,0.033,0.000,0.033,0.000
3,2026-09-02 03:00:00,143.654,0.000,0.000,0.020,0.033,0.000,0.033,0.000
4,2026-09-02 04:00:00,101.445,0.000,0.000,0.020,0.033,0.000,0.033,0.000
5,2026-09-02 05:00:00,63.013,0.000,0.000,0.020,0.034,0.000,0.034,0.000
6,2026-09-02 06:00:00,48.437,0.000,0.000,0.020,0.035,0.000,0.035,0.000
7,2026-09-02 07:00:00,25.197,0.000,0.000,0.020,0.037,0.000,0.038,0.001
8,2026-09-02 08:00:00,8.528,0.000,0.000,0.020,0.036,0.000,0.041,0.005
9,2026-09-02 09:00:00,35.625,0.000,0.000,0.020,0.028,0.000,0.046,0.018


In [8]:
con.close()
print("  fin")

  fin
